**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# HLS for FPGA: C to Gates

> ⚠️ **Draft — requires an HLS toolchain (Vitis HLS, free license) not available at authoring time.** An instructor should run each block before teaching; remove this banner after.

The [FPGA workshop](./Intro_FPGA.ipynb) wrote Verilog by hand; **high-level synthesis** compiles C/C++ into it — with #pragma annotations steering the hardware. You still think in [pipelines and timing](./Intro_FPGA.ipynb); you just stop hand-placing every register. The bridge course for software engineers entering hardware.

## 1. Pre-requisites

[Intro to FPGA](./Intro_FPGA.ipynb) (what synthesis produces), [Intro to C](../Intro_Programming/Intro_C.ipynb), [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) S1 (fixed point).

---
### 🕐 Session 1 of 2 — *C With Hardware Semantics* (~40 min)
**Goal:** arbitrary-precision types, the FIR again, and the pragmas that shape silicon.
**Builds on:** [Intro to FPGA](./Intro_FPGA.ipynb) S3. &nbsp; **Feeds into:** Session 2 (interfaces & integration).

---

💡 **Intuition.** HLS C is C *reinterpreted*: loops become pipelines, arrays become BRAMs, and `#pragma` lines are floor-plan instructions. The mental shift: you are not writing instructions to execute but *describing a datapath to instantiate* — the [FPGA workshop's](./Intro_FPGA.ipynb) lesson surviving the syntax change. The report, not the compiler exit code, is the real output: II (initiation interval — samples per clock), latency, and resource counts.

```cpp
// fir.cpp — the same Q15 FIR as [Intro_FPGA §4] and [Real_Time_DSP §1], in HLS C++
#include <ap_fixed.h>
typedef ap_fixed<16, 1> q15_t;              // 1 sign+int bit, 15 fraction: hardware Q1.15
typedef ap_fixed<34, 3> acc_t;              // accumulator with headroom — the rule, again

void fir(q15_t x_in, q15_t* y_out) {
    static const q15_t H[4] = {0.1, 0.4, 0.4, 0.1};
    static q15_t delay[4];
#pragma HLS ARRAY_PARTITION variable=delay complete   // registers, not BRAM: all taps at once
    acc_t acc = 0;
tap_loop:
    for (int k = 3; k > 0; k--) {
#pragma HLS UNROLL                                     // spatial: 4 multipliers, not 1 reused
        delay[k] = delay[k-1];
        acc += delay[k] * H[k];
    }
    delay[0] = x_in;
    acc += delay[0] * H[0];
    *y_out = (q15_t)acc;
}
// synthesis target: II=1 (one sample per clock). If the report says II>1, a dependency
// or resource limit broke the pipeline — the HLS debugging loop lives in that report.
```

**The C testbench IS the golden model** (the [FPGA workshop's](./Intro_FPGA.ipynb) verification
strategy, upgraded): the same `fir()` compiles for your laptop — assert bit-exact agreement with
a NumPy Q15 reference *before* synthesis, then let cosimulation replay it against the RTL.

---
### 🕐 Session 2 of 2 — *Interfaces & Integration* (~35 min)
**Goal:** AXI-Stream in, AXI-Lite control: dropping the block into a real system.
**Builds on:** Session 1.

---

```cpp
// streaming top-level: how the block meets the [SDR](../Intro_SDR/Software_Defined_Radio.ipynb)-
// style sample flow
#include <hls_stream.h>
void fir_stream(hls::stream<q15_t>& in, hls::stream<q15_t>& out, int n) {
#pragma HLS INTERFACE axis port=in
#pragma HLS INTERFACE axis port=out
#pragma HLS INTERFACE s_axilite port=n            // CPU pokes length/config over AXI-Lite
    for (int i = 0; i < n; i++) {
#pragma HLS PIPELINE II=1
        q15_t y; fir(in.read(), &y); out.write(y);
    }
}
```

💡 **Intuition.** Interfaces are where HLS projects live or die: `axis` streams match the
[producer/consumer](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) shape of sample pipelines,
`s_axilite` gives the CPU a control panel, and DMA moves buffers without the CPU touching
samples — the [Real-Time DSP](../Intro_DSP/Real_Time_DSP.ipynb) architecture, cast in silicon.
The classic pitfall table: unintended BRAM (missing partition pragma), II ruined by a
loop-carried dependency, and float sneaking in where `ap_fixed` was meant.

---
## Where next

- [Intro to FPGA](./Intro_FPGA.ipynb) — read the Verilog HLS emits; it demystifies both.
- [Sigma-Delta](../Intro_DSP/Sigma_Delta_Quantization.ipynb) — a decimating CIC in HLS is the perfect second project.